# Audit compression progress: the five reported numbers

This notebook computes the report defined in *Signed Compression Progress on a Sealed Audit is
Goodhart-Resistant*. Every quantity is in **nats**.

For a declared target distribution `Q` over audit units `z = (x, y)`, the population log loss of a
probability quote `p` is

```
L_Q(p) = mean over audit units of  -log p(y | x)
```

Given a benchmark-owned reference quote `p_ref`, an initial state `p_0` and a terminal state `p_T`,
the report is five numbers.

| # | Number | Definition |
|---|--------|------------|
| 1 | terminal level `B` | `L_Q(p_ref) - L_Q(p_T)` |
| 2 | signed gain `G` | `L_Q(p_0) - L_Q(p_T)` |
| 3 | slack `Gamma` | route-dependent; Section 4 |
| 4 | lower bound on improvement | `G - Gamma` |
| 5 | discrimination ratio | `G / Gamma` |

**What the lower bound licenses.** A run reporting signed gain `G` has improved on the declared
target `Q` by at least `G - Gamma` nats. That statement holds at every horizon and for every
adaptive trajectory covered by the declared event: no term in the number of training steps and no
term in the parameter count enters `Gamma`, so a longer run or a larger model is covered without
change. Because the increments are signed, the reported gain is already net of forgetting.

The statement is about `Q`, under the declared masking policy, in the declared unit. It is not a
forecast of behaviour on other work and carries no claim about safety, values, or coverage outside
`Q`. When `G` does not exceed `Gamma` the lower bound is not positive, and the statement available
is that no improvement is demonstrated at that resolution. The discrimination ratio `G / Gamma` is
dimensionless and comparable across substrates; it exceeds 1 exactly when the lower bound is
positive.

**Vocabulary.** `Gamma` is the *slack*. `G - Gamma` is the *lower bound on improvement*. The object
produced here is a bound.

**Interface.** Every function takes **arrays of per-audit-unit losses**, not model objects: three
numpy vectors of length `n` holding `-log p(y | x)` under `p_ref`, `p_0` and `p_T`. The core is
numpy-only and framework-agnostic. Section 11 shows, optionally, how to produce those vectors from a
torch model. Nothing in the core imports torch.

**Order of the notebook.** Sections 2 and 3 are the three checks that have to pass before any number
is computed: how the panel was used, what counts as one audit unit, and which estimator produced
the level. Sections 4 to 8 are the computation, the report object, panel sizing and the
reconciliation check. Section 9 reproduces the ARC-TGI run from the paper, asserts the published
values, and then prices what its archived metric costs; Section 10 shows the guardrail refusing
the other panel of that same run.


In [ ]:
import hashlib
import json
import math
import sys
import types
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

UNIT = "nats"

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("unit   :", UNIT)

## 1. Losses in, numbers out

The core takes three vectors of per-audit-unit losses over the *same* panel, in the same order:

- `loss_ref[i] = -log p_ref(y_i | x_i)` under the benchmark-owned reference quote,
- `loss_0[i]   = -log p_0(y_i | x_i)`   under the initial state,
- `loss_T[i]   = -log p_T(y_i | x_i)`   under the terminal state.

One entry per independent audit unit, and the panel level is their unweighted mean: see Section 3
for what changes when the level pools sub-units across records instead.

A finite-sample bound needs a bounded loss. The construction in the paper forms the full-support
quote `p_alpha = (1 - alpha) p + alpha u` with `u_min > 0`, so `0 <= -log p_alpha <= R` with
`R = -log(alpha * u_min)`. Declare that `R` and the checks below will hold the arrays to it. An
unfloored quote can produce an infinite loss on a single unit, which no concentration argument
covers.


In [ ]:
def check_loss_vector(x: Sequence[float], name: str, R: Optional[float] = None) -> np.ndarray:
    """Validate one per-audit-unit loss vector and return it as float64.

    Rejects the three mistakes that silently corrupt the report: log-likelihoods passed with the
    wrong sign, an unfloored quote that produced an infinite loss, and losses that exceed the
    declared range R used by the slack.
    """
    a = np.asarray(x, dtype=np.float64)
    if a.ndim != 1:
        raise ValueError(f"{name}: expected a 1-D vector of per-audit-unit losses, got shape {a.shape}")
    if a.size == 0:
        raise ValueError(f"{name}: empty loss vector")
    if not np.all(np.isfinite(a)):
        bad = int(np.sum(~np.isfinite(a)))
        raise ValueError(
            f"{name}: {bad} non-finite entries. -log p is infinite where the quote assigns zero "
            f"probability to the observed outcome; floor the quote before scoring."
        )
    if np.any(a < 0.0):
        raise ValueError(
            f"{name}: negative entries. These vectors hold -log p(y|x), which is non-negative; "
            f"a sign was dropped."
        )
    if R is not None and np.any(a > R + 1e-9):
        worst = float(a.max())
        raise ValueError(
            f"{name}: max entry {worst:.6f} exceeds the declared loss range R={R:.6f}. Either the "
            f"quote was not floored at the declared alpha, or R was declared too small."
        )
    return a


def aligned_panel(loss_ref, loss_0, loss_T, R: Optional[float] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Validate the three vectors and check they describe one panel, unit for unit."""
    a = check_loss_vector(loss_ref, "loss_ref", R)
    b = check_loss_vector(loss_0, "loss_0", R)
    c = check_loss_vector(loss_T, "loss_T", R)
    if not (a.size == b.size == c.size):
        raise ValueError(
            f"length mismatch: loss_ref={a.size}, loss_0={b.size}, loss_T={c.size}. The three "
            f"vectors must score the same panel, in the same order."
        )
    return a, b, c


def mean_log_loss(x, name: str = "loss", R: Optional[float] = None) -> float:
    """L_Q(p) estimated on the panel: the mean per-audit-unit log loss, in nats."""
    return float(check_loss_vector(x, name, R).mean())


_demo = np.full(8, math.log(4.0))
print("check_loss_vector on a constant quote of 1/4 over 8 units:")
print("  mean log loss = %.6f nats  (= ln 4 = %.6f)" % (mean_log_loss(_demo), math.log(4.0)))
try:
    check_loss_vector([0.3, -0.1, 0.2], "loss_0")
except ValueError as exc:
    print("  rejected as expected:", str(exc).split(".")[0])

## 2. The guardrail: how was the panel used?

Route (c) below prices a **fixed panel**. It is valid only if the panel produced no signal that
influenced training. If the panel's scores steered training in any way, the terminal state was
chosen with knowledge of the panel, the empirical loss at that state is no longer an unbiased read
of `L_Q`, and route (c) does not apply. Route (d) applies instead, with an honest count of the
audit-derived bits that reached training.

This is the most likely error, so it is a check and not a comment. `PanelProvenance` cannot be
built without answering all five channels, and route (c) raises on a panel that steered training.

The five channels, from the paper's accounting: audit scores entering a **reward** or training loss,
deciding **early stopping**, selecting a **checkpoint** to keep or report, selecting a
**hyperparameter**, schedule or architecture, or selecting or reweighting **training data**. The
paper charges scores, thresholds, selected examples, timing, stopping changes and scheduler changes
caused by audit feedback to the transcript.

Two properties of the declaration, both enforced below rather than asked for.

**It is immutable.** The channel answers are stored in a read-only mapping and the set of steering
channels is frozen at construction, so a declaration cannot be edited into a sealed one after the
fact.

**A steering panel released at least one bit.** A panel that changed training changed it one way
rather than another, so `transcript_bits` under 1 is refused. Without that floor, route (d) at
`B = 0` prices a fully steering panel *below* route (c) on a sealed one, and the cheapest slack in
the notebook would be the one bought by declaring the worst provenance.

In [ ]:
STEERING_CHANNELS: Tuple[str, ...] = (
    "reward",                     # audit scores entered a training reward or training loss
    "early_stopping",             # audit scores decided when training stopped
    "checkpoint_selection",       # audit scores chose which checkpoint was kept or reported
    "hyperparameter_selection",   # audit scores chose a hyperparameter, schedule or architecture
    "data_selection",             # audit scores chose or reweighted training data
)


class PanelSteeredError(RuntimeError):
    """Raised when a route is requested that the panel's provenance does not support."""


@dataclass(frozen=True)
class PanelProvenance:
    """A declaration of how an audit panel was used relative to training.

    Build this with declare_panel(), which makes every channel a required keyword argument.

    The declaration is immutable. influenced_training is replaced at construction by a read-only
    mapping and the steering set is frozen into a separate field, so sealed-ness is fixed at the
    moment of declaration and cannot be edited afterwards.
    """
    name: str
    influenced_training: Dict[str, bool]
    transcript_bits: Optional[float] = None
    note: str = ""
    steering: frozenset = field(default_factory=frozenset)

    def __post_init__(self) -> None:
        missing = [c for c in STEERING_CHANNELS if c not in self.influenced_training]
        if missing:
            raise ValueError(
                "declare every channel explicitly; missing: " + ", ".join(missing)
            )
        unknown = [k for k in self.influenced_training if k not in STEERING_CHANNELS]
        if unknown:
            raise ValueError("unknown channel(s): " + ", ".join(unknown))
        for k, v in self.influenced_training.items():
            if not isinstance(v, bool):
                raise TypeError(
                    f"channel {k!r} must be True or False, got {type(v).__name__}. A truthy "
                    f"placeholder is not a declaration."
                )
        # Freeze both views before any of them is read, so the declaration cannot be edited later.
        object.__setattr__(self, "steering",
                           frozenset(c for c in STEERING_CHANNELS if self.influenced_training[c]))
        object.__setattr__(self, "influenced_training",
                           types.MappingProxyType(dict(self.influenced_training)))
        if self.transcript_bits is not None and self.transcript_bits < 0:
            raise ValueError("transcript_bits must be non-negative")
        if self.steering:
            if self.transcript_bits is None:
                raise ValueError(
                    f"panel {self.name!r} steered training through "
                    f"{', '.join(self.steering_channels)}. Declare transcript_bits: an honest count "
                    f"of the audit-derived bits exposed to training, so route (d) can price it."
                )
            if float(self.transcript_bits) < 1.0:
                raise ValueError(
                    f"panel {self.name!r} steered training through "
                    f"{', '.join(self.steering_channels)}, so transcript_bits = "
                    f"{float(self.transcript_bits):g} is not an honest count. A panel that changed "
                    f"training released at least one bit: the run went one way rather than another. "
                    f"Count every score, threshold, selected example, timing signal, stopping "
                    f"change and scheduler change caused by audit feedback."
                )

    def _consistent_steering(self) -> frozenset:
        """The steering set, cross-checked against the channel answers on every read.

        The frozen field is redundant with influenced_training on purpose. A frozen dataclass can
        still be written through object.__setattr__, so holding the same fact twice makes any such
        edit visible: the two have to agree, and editing both consistently is declaring a different
        panel rather than slipping past the declaration.
        """
        derived = frozenset(c for c in STEERING_CHANNELS if self.influenced_training.get(c))
        if derived != self.steering:
            raise PanelSteeredError(
                f"panel {self.name!r} has been edited since it was declared: the channel answers "
                f"give {sorted(derived) or 'no steering'} and the frozen declaration says "
                f"{sorted(self.steering) or 'no steering'}. Declare the panel you mean with "
                f"declare_panel()."
            )
        return derived

    @property
    def steering_channels(self) -> Tuple[str, ...]:
        steering = self._consistent_steering()
        return tuple(c for c in STEERING_CHANNELS if c in steering)

    @property
    def sealed(self) -> bool:
        """True when the panel produced no signal that influenced training."""
        return not self._consistent_steering()

    @property
    def released_bits(self) -> float:
        """Cumulative audit-derived transcript bits: zero for a sealed panel, the declared count
        otherwise. This is the quantity the paper's public report carries."""
        if self.sealed:
            return 0.0
        return float(self.transcript_bits)

    def describe(self) -> str:
        if self.sealed:
            return f"panel {self.name!r}: sealed, no declared influence on training"
        return (
            f"panel {self.name!r}: steered training through "
            f"{', '.join(self.steering_channels)}; declared transcript {self.transcript_bits:g} bits"
        )


def declare_panel(
    name: str,
    *,
    reward: bool,
    early_stopping: bool,
    checkpoint_selection: bool,
    hyperparameter_selection: bool,
    data_selection: bool,
    transcript_bits: Optional[float] = None,
    note: str = "",
) -> PanelProvenance:
    """Declare how the panel was used. All five channels are required keyword arguments.

    Answer each with True or False for the run being reported. There is no default: a panel whose
    use has not been declared cannot be priced.
    """
    return PanelProvenance(
        name=name,
        influenced_training={
            "reward": reward,
            "early_stopping": early_stopping,
            "checkpoint_selection": checkpoint_selection,
            "hyperparameter_selection": hyperparameter_selection,
            "data_selection": data_selection,
        },
        transcript_bits=transcript_bits,
        note=note,
    )


sealed_example = declare_panel(
    "demo sealed panel",
    reward=False,
    early_stopping=False,
    checkpoint_selection=False,
    hyperparameter_selection=False,
    data_selection=False,
    note="drawn once, scored twice, never read during training",
)
steered_example = declare_panel(
    "demo steering probe",
    reward=True,
    early_stopping=False,
    checkpoint_selection=True,
    hyperparameter_selection=False,
    data_selection=False,
    transcript_bits=4096.0,
)
print(sealed_example.describe())
print(steered_example.describe())
print("released bits: sealed %g, steering %g"
      % (sealed_example.released_bits, steered_example.released_bits))

try:
    declare_panel(
        "undeclared",
        reward=True,
        early_stopping=False,
        checkpoint_selection=False,
        hyperparameter_selection=False,
        data_selection=False,
    )
except ValueError as exc:
    print("\nrefused, as expected:\n ", str(exc))

try:
    declare_panel(
        "zero-bit steering panel",
        reward=True, early_stopping=True, checkpoint_selection=True,
        hyperparameter_selection=True, data_selection=True,
        transcript_bits=0.0,
    )
except ValueError as exc:
    print("\nrefused, as expected:\n ", str(exc))

# The declaration is frozen: neither the field nor the mapping inside it can be edited.
_p = declare_panel("edit attempt", reward=True, early_stopping=False, checkpoint_selection=False,
                   hyperparameter_selection=False, data_selection=False, transcript_bits=64.0)
def _edit_through_object_setattr():
    object.__setattr__(_p, "steering", frozenset())
    return _p.sealed
for _attempt, _fn in (
    ("panel.influenced_training['reward'] = False",
     lambda: _p.influenced_training.__setitem__("reward", False)),
    ("panel.steering = frozenset()", lambda: setattr(_p, "steering", frozenset())),
    ("object.__setattr__(panel, 'steering', frozenset())", _edit_through_object_setattr),
):
    try:
        _fn()
        print("\nNOT REFUSED:", _attempt)
    except (TypeError, AttributeError, PanelSteeredError) as exc:
        print("\nrefused, as expected: %s -> %s" % (_attempt, type(exc).__name__))
object.__setattr__(_p, "steering", frozenset(["reward"]))       # undo the demonstration
print("still steering:", _p.steering_channels, "sealed =", _p.sealed)

## 3. What counts as one audit unit

`n` is the number of **independent** audit units. Cells within one grid, tokens within one document,
frames within one episode and turns within one dialogue are dependent, so the conservative unit is
the grid, the document, the episode, the dialogue. Substituting the token or cell count for `n`
shrinks the slack by roughly the square root of the number of sub-units per record and produces a
lower bound the panel does not support. Every route that consumes an `n` now runs this check on the
same call that computes the slack, and refuses a declared unit that is a sub-unit of a record.

**The declared unit also has to be the thing the estimator averages.** Route (c) prices the
*unweighted* mean of `n` per-unit losses, each in `[0, R]`; that is what Hoeffding at radius
`R*sqrt(ln(2N/delta)/(2n))` covers. An estimator that pools sub-units across records first --- total
loss over every cell in the panel divided by the total number of cells --- is a weighted mean
`sum_i w_i m_i` with `w_i = c_i / sum_j c_j` for sub-unit counts `c_i`. Its radius is

```
Delta_w = R * sqrt( sum_i w_i^2 * ln(2N/delta) / 2 ) = Delta * sqrt(n / n_eff),
n_eff = 1 / sum_i w_i^2 = (sum_i c_i)^2 / sum_i c_i^2
```

By Cauchy-Schwarz `sum_i w_i^2 >= 1/n`, so `n_eff <= n` and the unweighted radius is **never** the
conservative one for a weighted estimator: the error has a fixed sign. `effective_n()` below
computes `n_eff` from the sub-unit counts, and `panel_radius()` demands it whenever a weighted
estimator is declared.

Two of the artifacts in this repository sit on opposite sides of that line, and both are named here
because the difference is not visible in the reported number.

- `examples/lambada-openai/` is the unweighted case and is what this interface expects: one value
  per passage, averaged over that passage's tokens, then averaged unweighted over the 5,153
  passages. `Delta` is exactly right there.
- The archived ARC-TGI audit metric is the weighted case. `audit_panel_loss` in
  `experiments/cp_models.py` returns total floored cross-entropy over every content cell of the
  panel divided by the total content-cell count; it computes no per-item value. ARC-TGI grids run
  from 1x1 to 30x30 inside a padded 30x30 canvas, so the weights genuinely vary. Section 9 prices
  what that costs.

In [ ]:
DEPENDENT_UNIT_KINDS = {
    "token", "subword", "wordpiece", "character", "char", "byte",
    "cell", "grid_cell", "pixel", "voxel", "patch",
    "frame", "timestep", "step", "turn", "utterance", "sentence", "line",
}
INDEPENDENT_UNIT_KINDS = {
    "example", "document", "item", "episode", "trajectory", "question",
    "problem", "task_instance", "grid", "dialogue", "record", "audit_unit",
    "passage", "panel_item",
}

# How the panel level was estimated from the per-record losses. This is a declaration, not an
# inference: the two differ by a factor the reported number cannot reveal.
UNWEIGHTED_MEAN = "unweighted_mean_of_unit_losses"
WEIGHTED_MEAN = "weighted_mean_of_unit_losses"
ESTIMATOR_KINDS = (UNWEIGHTED_MEAN, WEIGHTED_MEAN)
_ESTIMATOR_ALIASES = {
    "unweighted": UNWEIGHTED_MEAN, "mean": UNWEIGHTED_MEAN, "mean_of_unit_means": UNWEIGHTED_MEAN,
    UNWEIGHTED_MEAN: UNWEIGHTED_MEAN,
    "weighted": WEIGHTED_MEAN, "ratio": WEIGHTED_MEAN, "pooled_ratio": WEIGHTED_MEAN,
    "cell_weighted_ratio": WEIGHTED_MEAN, "token_weighted_ratio": WEIGHTED_MEAN,
    "content_cell_weighted_ratio": WEIGHTED_MEAN, WEIGHTED_MEAN: WEIGHTED_MEAN,
}


class DependentUnitError(ValueError):
    """Raised when a slack is priced at a count of sub-units rather than independent records."""


class WeightedEstimatorError(ValueError):
    """Raised when a weighted panel level is priced with the unweighted radius."""


def normalise_unit_kind(unit_kind: str) -> str:
    return str(unit_kind).strip().lower().replace("-", "_").replace(" ", "_")


def normalise_estimator(estimator: str) -> str:
    key = normalise_unit_kind(estimator)
    if key not in _ESTIMATOR_ALIASES:
        raise ValueError(
            "unknown estimator %r; declare %r (one value per record, averaged unweighted) or %r "
            "(sub-units pooled across records, e.g. total loss over total cells)"
            % (estimator, UNWEIGHTED_MEAN, WEIGHTED_MEAN)
        )
    return _ESTIMATOR_ALIASES[key]


def effective_n(sub_unit_counts) -> float:
    """n_eff = (sum c_i)^2 / sum c_i^2 for a panel level weighted by sub-unit counts.

    c_i is the number of scored sub-units (cells, tokens, frames) in record i. n_eff equals n when
    the counts are equal and falls below it otherwise; it is the sample size the Hoeffding radius
    of a weighted mean actually has.
    """
    c = np.asarray(sub_unit_counts, dtype=np.float64).reshape(-1)
    if c.size == 0:
        raise ValueError("sub_unit_counts is empty")
    if np.any(c < 0) or not np.all(np.isfinite(c)):
        raise ValueError("sub-unit counts must be finite and non-negative")
    s = float(c.sum())
    if s <= 0:
        raise ValueError("sub-unit counts sum to zero")
    return s * s / float((c * c).sum())


def check_independent_units(
    n_units: int,
    unit_kind: str,
    *,
    n_records: Optional[int] = None,
    strict: bool = False,
    quiet: bool = False,
) -> List[str]:
    """Warn when the declared n looks like a sub-unit count rather than an independent-unit count.

    n_records, when given, is the number of independent records (examples, documents, episodes)
    the panel was drawn from. Returns the list of warnings and prints them; strict=True raises
    instead.
    """
    warnings_out: List[str] = []
    kind = normalise_unit_kind(unit_kind)
    if int(n_units) < 1:
        raise ValueError("n_units must be a positive integer")

    if kind in DEPENDENT_UNIT_KINDS:
        warnings_out.append(
            f"declared unit {kind!r} is a sub-unit of an example, document or episode. Sub-units "
            f"within one record are dependent, so n={n_units} is anti-conservative. Use the number "
            f"of independent records instead."
        )
    elif kind not in INDEPENDENT_UNIT_KINDS:
        warnings_out.append(
            f"unrecognised unit kind {kind!r}. Confirm it is the independent unit: if several of "
            f"them come from one example, document or episode, they are dependent."
        )

    if n_records is not None:
        if int(n_records) < 1:
            raise ValueError("n_records must be a positive integer")
        if n_units > n_records:
            ratio = n_units / float(n_records)
            factor = math.sqrt(ratio)
            warnings_out.append(
                f"n_units={n_units} exceeds n_records={n_records} at {ratio:.1f} sub-units per "
                f"record. The conservative n is {n_records}; using {n_units} understates the slack "
                f"by about a factor of {factor:.1f}."
            )

    if kind in INDEPENDENT_UNIT_KINDS and n_units >= 1_000_000:
        warnings_out.append(
            f"n={n_units} independent {kind}s is a large audit panel. Confirm these are records "
            f"and not tokens, cells or frames."
        )

    if warnings_out:
        if strict:
            raise ValueError("independent-unit check failed:\n  - " + "\n  - ".join(warnings_out))
        for w in warnings_out:
            print("WARNING (independent unit): " + w)
    elif not quiet:
        print(f"independent-unit check passed: n={n_units} {kind}s")
    return warnings_out


def assert_independent_units(n_units: int, unit_kind: str, *,
                             n_records: Optional[int] = None, where: str = "this slack",
                             quiet: bool = False) -> str:
    """The pricing-path form of the check: refuse a sub-unit kind, warn on anything unrecognised.

    Every route that consumes an n calls this, so the declared unit is checked on the same call
    that computes the slack rather than in a separate demonstration.
    """
    kind = normalise_unit_kind(unit_kind)
    if kind in DEPENDENT_UNIT_KINDS:
        raise DependentUnitError(
            f"{where} was priced at n={n_units} {kind}s. A {kind} is a sub-unit of an example, "
            f"document or episode, and sub-units inside one record are not independent draws, so "
            f"this n shrinks the slack by roughly the square root of the number of {kind}s per "
            f"record and buys a lower bound the panel does not support. Price it at the number of "
            f"independent records and declare that unit."
        )
    check_independent_units(n_units, kind, n_records=n_records, quiet=quiet)
    return kind


def pricing_sample_size(*, n: int, unit_kind: str, n_records: Optional[int],
                        estimator: str, n_effective: Optional[float],
                        where: str, quiet: bool = False) -> Tuple[float, str, float]:
    """Validate the declared unit and estimator; return (sample size for the radius, estimator, n_eff).

    An unweighted mean is priced at n. A weighted one is priced at n_eff, which has to be declared:
    it is not recoverable from the reported level.
    """
    assert_independent_units(n, unit_kind, n_records=n_records, where=where, quiet=quiet)
    est = normalise_estimator(estimator)
    if est == UNWEIGHTED_MEAN:
        if n_effective is not None and abs(float(n_effective) - float(n)) > 1e-9:
            raise ValueError(
                "an unweighted mean of per-unit losses has n_eff = n; n_effective=%r contradicts "
                "the declared estimator. Declare %r if the panel level pools sub-units."
                % (n_effective, WEIGHTED_MEAN))
        return float(n), est, float(n)
    if n_effective is None:
        raise WeightedEstimatorError(
            f"{where} declares a {WEIGHTED_MEAN}: the panel level pools sub-units across records "
            f"(total loss over total sub-units), so it is sum_i w_i m_i with data-dependent "
            f"weights, not the unweighted mean of {n} bounded terms that this radius covers. Its "
            f"radius is Delta * sqrt(n / n_eff) with n_eff = (sum_i c_i)^2 / sum_i c_i^2, and "
            f"n_eff <= n always, so the unweighted radius is never the conservative one here. "
            f"Compute n_eff from the per-record sub-unit counts with effective_n() and pass it as "
            f"n_effective, or change the metric to the unweighted mean of per-record losses."
        )
    ne = float(n_effective)
    if not math.isfinite(ne) or ne <= 0:
        raise ValueError("n_effective must be a positive finite number")
    if ne > float(n) + 1e-9:
        raise ValueError(
            "n_effective=%.6g exceeds n=%d. n_eff = (sum c_i)^2 / sum c_i^2 is at most the number "
            "of records, with equality only when every record carries the same number of "
            "sub-units." % (ne, n))
    return ne, est, ne


print("-- correct declaration for the ARC-TGI sealed panel:")
check_independent_units(12288, "audit_unit", n_records=12288)
print()
print("-- the same panel declared by its cell count:")
check_independent_units(12288 * 900, "cell", n_records=12288)
print()
print("-- the same declaration on the pricing path, where it is a refusal:")
try:
    assert_independent_units(12288 * 900, "cell", n_records=12288, where="a route (c) slack")
except DependentUnitError as exc:
    print("refused:", str(exc).split(". ")[0] + ".")
print()
print("-- effective sample size of a weighted panel level:")
_rng_c = np.random.default_rng(3)
for _label, _counts in (("every record 100 cells", np.full(12288, 100.0)),
                        ("sides uniform on 3..12", _rng_c.integers(3, 13, 12288) ** 2),
                        ("sides uniform on 1..30", _rng_c.integers(1, 31, 12288) ** 2)):
    _ne = effective_n(_counts)
    print("  %-24s n_eff = %9.1f   radius inflation sqrt(n/n_eff) = %.3f"
          % (_label, _ne, math.sqrt(12288 / _ne)))

## 4. The four slack routes

The slack is the additive error budget in the endpoint inequality. The user declares which route
applies; the notebook computes only the matching formula. Every route that consumes a count of audit
units also requires the **unit kind** and the **estimator**, and checks both before it returns a
number.

**(a) Complete verifier, `eta`-complete.**

```
Gamma = 2 * eta
```

**(b) Fresh endpoints.** Independent batches of size `m0` and `mT` at the two endpoints, with loss
widths `W0` and `WT`, and `delta0 + deltaT <= delta`:

```
Gamma = W0 * sqrt(ln(2/delta0) / (2*m0)) + WT * sqrt(ln(2/deltaT) / (2*mT))
```

**(c) Fixed panel.** `n` units, loss range `R`, a declared finite candidate set of size `N`:

```
Delta = R * sqrt(ln(2*N/delta) / (2*n))
Gamma = 2 * Delta
```

The two-sided form covers an upper deviation at `p_0` and a lower deviation at `p_T`. Valid only for
a panel that produced no signal that influenced training.

**(d) Reused panel** whose audit-derived transcript takes at most `2^B` values:

```
Gamma = R * sqrt(2 * (B*ln(2) + ln(1/delta)) / n)
```

Route (d) is where a steered panel goes. `B` is a declared budget on the released transcript, not an
estimate of how much information the training run actually used.

**Weighted panel levels.** Routes (c) and (d) both put `n` under a square root as the sample size of
an unweighted mean. When the declared estimator is a weighted mean, `n` is replaced by `n_eff`
throughout, which enlarges both slacks by `sqrt(n / n_eff)`. Declaring a weighted estimator without
`n_eff` raises rather than falling back on the unweighted radius.

**Two things a `Slack` carries besides its value.** It carries the `PanelProvenance` it was priced
under, so a route (c) or (d) slack cannot be assembled by hand around the guardrail, and it carries
the estimator it prices, so a report can tell whether the slack covers the levels it is printed next
to.

In [ ]:
@dataclass(frozen=True)
class Slack:
    """A slack value together with the route, provenance and parameters that produced it.

    Routes (c) and (d) require the PanelProvenance they were priced under. Carrying it here is what
    stops a hand-assembled Slack from reaching five_numbers() with the guardrail never consulted:
    the check lives in the object the report consumes, not only in the function that builds it.
    """
    gamma: float
    route: str
    route_label: str
    params: Dict[str, Any]
    transcript_bits: Optional[float] = None
    provenance: Optional[PanelProvenance] = None
    estimator: str = UNWEIGHTED_MEAN

    def __post_init__(self) -> None:
        object.__setattr__(self, "params", types.MappingProxyType(dict(self.params)))
        if self.route in ("c", "d"):
            if not isinstance(self.provenance, PanelProvenance):
                raise PanelSteeredError(
                    f"a route ({self.route}) slack has to carry the PanelProvenance it was priced "
                    f"under. Build it with compute_slack(); a Slack assembled by hand is a number "
                    f"with a label on it, not a declaration."
                )
            if self.route == "c" and not self.provenance.sealed:
                raise PanelSteeredError(
                    f"route (c) slack built on panel {self.provenance.name!r}, which steered "
                    f"training through {', '.join(self.provenance.steering_channels)}."
                )

    def __str__(self) -> str:
        return f"Gamma = {self.gamma:.6f} nats  [route {self.route}: {self.route_label}]"


def slack_complete_verifier(*, eta: float) -> Slack:
    """Route (a): a complete verifier, eta-complete. Gamma = 2*eta."""
    if eta < 0:
        raise ValueError("eta must be non-negative")
    return Slack(2.0 * eta, "a", "complete verifier, eta-complete", {"eta": float(eta)})


def slack_fresh_endpoints(*, m0: int, mT: int, W0: float, WT: float,
                          delta0: float, deltaT: float, delta: float,
                          unit_kind: str, n_records_0: Optional[int] = None,
                          n_records_T: Optional[int] = None) -> Slack:
    """Route (b): independent fresh batches at the two endpoints."""
    for nm, v in (("m0", m0), ("mT", mT)):
        if int(v) < 1:
            raise ValueError(f"{nm} must be a positive integer")
    for nm, v in (("W0", W0), ("WT", WT)):
        if v <= 0:
            raise ValueError(f"{nm} must be positive")
    for nm, v in (("delta0", delta0), ("deltaT", deltaT), ("delta", delta)):
        if not 0.0 < v < 1.0:
            raise ValueError(f"{nm} must lie in (0, 1)")
    if delta0 + deltaT > delta + 1e-15:
        raise ValueError(
            f"delta0 + deltaT = {delta0 + deltaT:g} exceeds delta = {delta:g}; the two endpoint "
            f"confidences must fit inside the declared budget."
        )
    assert_independent_units(m0, unit_kind, n_records=n_records_0,
                             where="the initial batch of a route (b) slack")
    assert_independent_units(mT, unit_kind, n_records=n_records_T,
                             where="the terminal batch of a route (b) slack")
    term0 = W0 * math.sqrt(math.log(2.0 / delta0) / (2.0 * m0))
    termT = WT * math.sqrt(math.log(2.0 / deltaT) / (2.0 * mT))
    return Slack(
        term0 + termT, "b", "fresh endpoints",
        {"m0": int(m0), "mT": int(mT), "W0": float(W0), "WT": float(WT),
         "delta0": float(delta0), "deltaT": float(deltaT), "delta": float(delta),
         "unit_kind": normalise_unit_kind(unit_kind),
         "term_initial": term0, "term_terminal": termT},
    )


def panel_radius(*, n: int, R: float, delta: float, N: float,
                 unit_kind: str, n_records: Optional[int] = None,
                 estimator: str = UNWEIGHTED_MEAN, n_effective: Optional[float] = None,
                 where: str = "a route (c) radius", quiet: bool = False) -> float:
    """Route (c) per-endpoint radius: Delta = R*sqrt(ln(2N/delta)/(2n)).

    unit_kind is required: n cannot be priced without saying what it counts. A weighted estimator
    is priced at n_effective in place of n.
    """
    if int(n) < 1:
        raise ValueError("n must be a positive integer")
    if R <= 0:
        raise ValueError("R must be positive")
    if not 0.0 < delta < 1.0:
        raise ValueError("delta must lie in (0, 1)")
    if N < 1:
        raise ValueError("N must be at least 1")
    n_price, _est, _ne = pricing_sample_size(
        n=n, unit_kind=unit_kind, n_records=n_records,
        estimator=estimator, n_effective=n_effective, where=where, quiet=quiet)
    return R * math.sqrt(math.log(2.0 * N / delta) / (2.0 * n_price))


ROUTE_C_REFUSAL = """Route (c) is not available for panel {name!r}.

The panel produced signal that influenced training through: {channels}.
A panel whose scores steered training is not a fixed-model measurement: the terminal state was
chosen with knowledge of the panel, so its empirical loss there is not an unbiased read of L_Q and
the uniform-deviation event that route (c) prices does not cover it.

Use route (d) with an honest bit count of the audit-derived transcript exposed to training:

    compute_slack("d", provenance=prov, n={n}, R={R!r}, delta={delta!r}, unit_kind=...,
                  transcript_bits=B)

Count every score, threshold, selected example, timing signal, stopping change and scheduler change
caused by audit feedback. If the released bits cannot be counted, the panel does not support a lower
bound at all."""


def slack_fixed_panel(*, n: int, R: float, delta: float, N: float,
                      provenance: PanelProvenance, unit_kind: str,
                      n_records: Optional[int] = None,
                      estimator: str = UNWEIGHTED_MEAN,
                      n_effective: Optional[float] = None,
                      quiet: bool = False) -> Slack:
    """Route (c): fixed panel, Gamma = 2*Delta. Refuses a panel that steered training."""
    if not isinstance(provenance, PanelProvenance):
        raise PanelSteeredError(
            "route (c) requires a declared PanelProvenance. Build one with declare_panel() and "
            "answer all five steering channels."
        )
    if not provenance.sealed:
        raise PanelSteeredError(ROUTE_C_REFUSAL.format(
            name=provenance.name, channels=", ".join(provenance.steering_channels),
            n=n, R=R, delta=delta,
        ))
    n_price, est, ne = pricing_sample_size(
        n=n, unit_kind=unit_kind, n_records=n_records, estimator=estimator,
        n_effective=n_effective, where="a route (c) slack", quiet=quiet)
    Delta = R * math.sqrt(math.log(2.0 * N / delta) / (2.0 * n_price))
    return Slack(
        2.0 * Delta, "c", "fixed panel, declared candidate set",
        {"n": int(n), "R": float(R), "delta": float(delta), "N": float(N), "Delta": Delta,
         "panel": provenance.name, "unit_kind": normalise_unit_kind(unit_kind),
         "estimator": est, "n_effective": ne, "n_priced": n_price},
        provenance=provenance, estimator=est,
    )


def slack_reused_panel(*, n: int, R: float, delta: float, transcript_bits: float,
                       provenance: PanelProvenance, unit_kind: str,
                       n_records: Optional[int] = None,
                       estimator: str = UNWEIGHTED_MEAN,
                       n_effective: Optional[float] = None,
                       quiet: bool = False) -> Slack:
    """Route (d): reused panel with a transcript taking at most 2^B values."""
    if int(n) < 1:
        raise ValueError("n must be a positive integer")
    if R <= 0:
        raise ValueError("R must be positive")
    if not 0.0 < delta < 1.0:
        raise ValueError("delta must lie in (0, 1)")
    if transcript_bits is None or transcript_bits < 0:
        raise ValueError("transcript_bits must be a declared non-negative bit budget")
    if not isinstance(provenance, PanelProvenance):
        raise PanelSteeredError(
            "route (d) requires a declared PanelProvenance. Build one with declare_panel() and "
            "answer all five steering channels."
        )
    B = float(transcript_bits)
    if not provenance.sealed and B < 1.0:
        raise ValueError(
            f"panel {provenance.name!r} steered training through "
            f"{', '.join(provenance.steering_channels)}, so a budget of {B:g} bits is not an "
            f"honest count; a panel that changed training released at least one bit."
        )
    if not provenance.sealed and B + 1e-9 < provenance.released_bits:
        print("WARNING (transcript): pricing at B = %g bits, below the %g bits declared for panel "
              "%r. A budget under the declared count is a hypothetical, not this run's slack."
              % (B, provenance.released_bits, provenance.name))
    n_price, est, ne = pricing_sample_size(
        n=n, unit_kind=unit_kind, n_records=n_records, estimator=estimator,
        n_effective=n_effective, where="a route (d) slack", quiet=quiet)
    gamma = R * math.sqrt(2.0 * (B * math.log(2.0) + math.log(1.0 / delta)) / n_price)
    return Slack(
        gamma, "d", "reused panel, bounded audit transcript",
        {"n": int(n), "R": float(R), "delta": float(delta), "transcript_bits": B,
         "unit_kind": normalise_unit_kind(unit_kind), "estimator": est,
         "n_effective": ne, "n_priced": n_price, "panel": provenance.name},
        transcript_bits=B, provenance=provenance, estimator=est,
    )


_ROUTES = {
    "a": slack_complete_verifier,
    "b": slack_fresh_endpoints,
    "c": slack_fixed_panel,
    "d": slack_reused_panel,
}


def compute_slack(route: str, *, provenance: Optional[PanelProvenance] = None, **params) -> Slack:
    """Declare the route and its parameters; returns the matching slack.

    provenance is required for routes (c) and (d). Route (c) refuses a panel that steered training.
    """
    key = str(route).strip().lower().lstrip("(").rstrip(")")
    if key not in _ROUTES:
        raise ValueError(f"unknown route {route!r}; declare one of 'a', 'b', 'c', 'd'")
    if key == "c":
        return slack_fixed_panel(provenance=provenance, **params)
    if key == "d":
        if "transcript_bits" not in params:
            if provenance is None or provenance.transcript_bits is None:
                raise ValueError(
                    "route (d) needs transcript_bits, either as an argument or on the provenance"
                )
            params["transcript_bits"] = provenance.transcript_bits
        return slack_reused_panel(provenance=provenance, **params)
    return _ROUTES[key](**params)


print(compute_slack("a", eta=0.02))
print(compute_slack("b", m0=4096, mT=4096, W0=4.605170186, WT=4.605170186,
                    delta0=0.025, deltaT=0.025, delta=0.05, unit_kind="example"))
print(compute_slack("c", provenance=sealed_example, n=12288, R=4.605170186, delta=0.05, N=1,
                    unit_kind="audit_unit"))
print(compute_slack("d", provenance=steered_example, n=12288, R=4.605170186, delta=0.05,
                    unit_kind="audit_unit"))
print()
print("-- the same panel priced at its cell count, on the pricing path:")
try:
    compute_slack("c", provenance=sealed_example, n=12288 * 900, R=4.605170186, delta=0.05, N=1,
                  unit_kind="cell", n_records=12288)
except DependentUnitError as exc:
    print("refused:", str(exc).split(". ")[0] + ".")
print()
print("-- a weighted panel level priced with the unweighted radius:")
try:
    compute_slack("c", provenance=sealed_example, n=12288, R=4.605170186, delta=0.05, N=1,
                  unit_kind="audit_unit", estimator=WEIGHTED_MEAN)
except WeightedEstimatorError as exc:
    print("refused:", str(exc).split(". ")[0] + ".")
print("  same level with n_eff declared:",
      compute_slack("c", provenance=sealed_example, n=12288, R=4.605170186, delta=0.05, N=1,
                    unit_kind="audit_unit", estimator=WEIGHTED_MEAN, n_effective=8400.0))
print()
print("-- a route (c) Slack assembled by hand, around compute_slack():")
try:
    Slack(0.154891, "c", "fixed panel, declared candidate set", {"n": 12288})
except PanelSteeredError as exc:
    print("refused:", str(exc).split(". ")[0] + ".")

## 5. The five numbers

`five_numbers()` takes the three loss vectors and a `Slack`, and returns the report quantities. It
is the whole computation: everything else in this notebook is a check on its inputs or a way of
publishing its outputs.


In [ ]:
@dataclass(frozen=True)
class FiveNumbers:
    """The five reported numbers, all in nats except the dimensionless ratio."""
    terminal_level: float          # B = L_Q(p_ref) - L_Q(p_T)
    signed_gain: float             # G = L_Q(p_0) - L_Q(p_T)
    slack: float                   # Gamma
    lower_bound: float             # G - Gamma
    discrimination_ratio: float    # G / Gamma
    level_ref: float
    level_initial: float
    level_terminal: float
    n_units: int
    route: str

    def summary(self) -> str:
        lines = [
            f"  1. terminal level        B = {self.terminal_level:+.4f} nats",
            f"  2. signed gain           G = {self.signed_gain:+.4f} nats",
            f"  3. slack             Gamma = {self.slack:.6f} nats   [route {self.route}]",
            f"  4. lower bound   G - Gamma = {self.lower_bound:+.4f} nats",
            f"  5. ratio         G / Gamma = {self.discrimination_ratio:.2f}",
        ]
        if self.lower_bound > 0.0:
            lines.append(
                f"     -> improved by at least {self.lower_bound:.4f} nats on the declared target"
            )
        else:
            lines.append(
                "     -> no improvement is demonstrated at this resolution"
            )
        return "\n".join(lines)


def five_numbers(loss_ref, loss_0, loss_T, slack: Slack, *, R: Optional[float] = None) -> FiveNumbers:
    """Compute the five reported numbers from three per-audit-unit loss vectors and a slack."""
    if not isinstance(slack, Slack):
        raise TypeError("slack must be a Slack; build one with compute_slack()")
    a, b, c = aligned_panel(loss_ref, loss_0, loss_T, R)
    if slack.route == "c" and not slack.provenance.sealed:
        raise PanelSteeredError(
            f"route (c) slack on panel {slack.provenance.name!r}, which steered training through "
            f"{', '.join(slack.provenance.steering_channels)}.")
    declared_n = slack.params.get("n")
    if declared_n is not None and int(declared_n) != int(a.size):
        raise ValueError(
            f"the slack was priced at n = {int(declared_n)} but the loss vectors hold {int(a.size)} "
            f"units. The slack and the evidence have to describe the same panel; a slack priced at "
            f"one n and printed beside another is the token-count error with an extra step."
        )
    L_ref, L_0, L_T = float(a.mean()), float(b.mean()), float(c.mean())
    B = L_ref - L_T
    G = L_0 - L_T
    gamma = float(slack.gamma)
    lower = G - gamma
    if gamma > 0.0:
        ratio = G / gamma
    else:
        ratio = math.inf if G > 0.0 else math.nan
    return FiveNumbers(
        terminal_level=B, signed_gain=G, slack=gamma, lower_bound=lower,
        discrimination_ratio=ratio, level_ref=L_ref, level_initial=L_0, level_terminal=L_T,
        n_units=int(a.size), route=slack.route,
    )


_n = 4096
_rng = np.random.default_rng(0)
_ref = np.full(_n, math.log(10.0))
_l0 = np.clip(_rng.normal(2.10, 0.30, _n), 0.0, 4.605170186)
_lT = np.clip(_rng.normal(1.60, 0.30, _n), 0.0, 4.605170186)
_prov = declare_panel("toy sealed panel", reward=False, early_stopping=False,
                      checkpoint_selection=False, hyperparameter_selection=False,
                      data_selection=False)
_slack = compute_slack("c", provenance=_prov, n=_n, R=4.605170186, delta=0.05, N=1,
                       unit_kind="example")
print("toy panel, n = %d" % _n)
print(five_numbers(_ref, _l0, _lT, _slack, R=4.605170186).summary())
print()
_wrong_n = compute_slack("c", provenance=_prov, n=_n * 64, R=4.605170186, delta=0.05, N=1,
                         unit_kind="example")
try:
    five_numbers(_ref, _l0, _lT, _wrong_n, R=4.605170186)
except ValueError as exc:
    print("slack and evidence must agree on n -- refused:", str(exc).split(". ")[0] + ".")

## 6. The report object

The paper's Discussion fixes what a public report contains: the terminal level against the benchmark
reference, the cumulative training gain, the target and masking-policy versions, the primitive audit
unit, the confidence or completeness slack, the cumulative audit-derived transcript bits, and the
commitment record. `audit_cp_report()` emits exactly those fields as JSON, plus the two derived
numbers and a digest that binds the report to the loss vectors it was computed from.

Four of those fields are checked rather than copied.

- **The primitive audit unit** is read for the kind it names and refused when that kind is a
  sub-unit, and `n_independent_units` has to match both the length of the loss vectors and the `n`
  the slack was priced at. A slack priced at one `n` and published beside another is the token-count
  error with an extra step between the mistake and the number.
- **The cumulative audit-derived transcript bits** is a count, so it is emitted as a number for
  every panel-based route: zero for a sealed panel, the declared budget for a steered one. Absent is
  not a value it can take.
- **The estimator** is declared alongside the unit. When the slack does not price the estimator that
  produced the levels, the report still publishes the levels and the gain, and publishes no lower
  bound: `bound_status` says why, and `n_effective_required_for_positive_bound` says what would have
  to hold for the sign to survive.
- **The commitment record** is the protocol's second step: the state is committed before the
  randomness that determines its challenge is drawn. Supply the record you actually hold. A digest
  of the scored losses is refused, because it can only be computed after scoring and therefore
  commits to nothing. When no commitment was recorded, say so and take the warning.

In [ ]:
def evidence_digest(loss_ref, loss_0, loss_T) -> str:
    """sha256 over the three loss vectors, so a report is bound to the evidence it came from."""
    a, b, c = aligned_panel(loss_ref, loss_0, loss_T)
    h = hashlib.sha256()
    for name, arr in (("ref", a), ("p0", b), ("pT", c)):
        h.update(name.encode("utf-8"))
        h.update(np.ascontiguousarray(arr, dtype="<f8").tobytes())
    return "sha256:" + h.hexdigest()


REQUIRED_COMMITMENT_KEYS = ("state_commitment", "committed_at", "challenge_drawn_after_commitment")


def _finite_or_none(x: float) -> Optional[float]:
    return float(x) if math.isfinite(x) else None


def _words(text: str) -> List[str]:
    out, cur = [], []
    for ch in str(text).lower():
        if ch.isalnum() or ch == "_":
            cur.append(ch)
        elif cur:
            out.append("".join(cur))
            cur = []
    if cur:
        out.append("".join(cur))
    return out


def audit_unit_kind(audit_unit: str) -> Optional[str]:
    """The unit kind named in the first clause of an audit_unit description, if it names one."""
    whole = normalise_unit_kind(audit_unit)
    if whole in DEPENDENT_UNIT_KINDS or whole in INDEPENDENT_UNIT_KINDS:
        return whole
    head = str(audit_unit).split(";")[0].split(",")[0]
    for w in _words(head):
        if w in DEPENDENT_UNIT_KINDS or w in INDEPENDENT_UNIT_KINDS:
            return w
    return None


def n_effective_for_positive_bound(*, gain: float, R: float, delta: float, N: float) -> Optional[int]:
    """Smallest n_eff at which a route (c) slack still sits below the reported gain."""
    if gain <= 0:
        return None
    return int(math.ceil(2.0 * R * R * math.log(2.0 * N / delta) / (gain * gain)))


def audit_cp_report(
    loss_ref, loss_0, loss_T, slack: Slack,
    *,
    target_version: str,
    reference_version: str,
    masking_policy_version: str,
    audit_unit: str,
    commitment_record: Dict[str, Any],
    estimator: str = UNWEIGHTED_MEAN,
    R: Optional[float] = None,
    n_independent_units: Optional[int] = None,
    n_effective: Optional[float] = None,
    component_notes: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """Compute the five numbers and emit the public report object."""
    missing = [k for k in REQUIRED_COMMITMENT_KEYS if k not in commitment_record]
    if missing:
        raise ValueError(
            "commitment_record is missing " + ", ".join(missing) +
            ". The report is the scalar plus the evidence process that gives it meaning."
        )
    digest = evidence_digest(loss_ref, loss_0, loss_T)
    commitment = commitment_record.get("state_commitment")
    if isinstance(commitment, str) and commitment.strip() == digest:
        raise ValueError(
            "state_commitment is the evidence digest of the loss vectors this report was computed "
            "from. That hash exists only after the state was scored, so it commits to nothing: it "
            "cannot have been published before the challenge was drawn. The commitment record has "
            "to name something fixed beforehand -- a checkpoint hash, a published model revision, a "
            "signed manifest -- or say that none was recorded."
        )
    if commitment is None or (isinstance(commitment, str) and not commitment.strip()):
        print(
            "WARNING (commitment): no state commitment recorded. The endpoint bound assumes the "
            "audited state was fixed before its challenge was drawn, and nothing in this report "
            "witnesses that."
        )
    if commitment_record["challenge_drawn_after_commitment"] is not True:
        print(
            "WARNING (commitment): challenge randomness was not drawn after the state was "
            "committed. The endpoint bound assumes the audited state was fixed before its "
            "challenge was drawn."
        )

    unit_kind = audit_unit_kind(audit_unit)
    if unit_kind is None:
        print("WARNING (audit unit): %r names no recognised unit kind. Say what one audit unit is: "
              "an example, a document, a passage, an episode." % audit_unit)
    elif unit_kind in DEPENDENT_UNIT_KINDS:
        raise DependentUnitError(
            "audit_unit %r names a %s, which is a sub-unit of an example, document or episode. "
            "Sub-units inside one record are not independent draws, so a report published with "
            "that unit claims a resolution the panel does not have." % (audit_unit, unit_kind)
        )

    fn = five_numbers(loss_ref, loss_0, loss_T, slack, R=R)
    n_units = int(n_independent_units if n_independent_units is not None else fn.n_units)
    if n_units != fn.n_units:
        raise ValueError(
            "n_independent_units = %d but the loss vectors hold %d units. The published unit count "
            "has to be the one the levels were computed over." % (n_units, fn.n_units))
    slack_n = slack.params.get("n")
    if slack_n is not None and int(slack_n) != n_units:
        raise ValueError(
            "the slack was priced at n = %d and the report publishes n = %d. One object cannot "
            "carry a slack bought at one panel size next to another." % (int(slack_n), n_units))

    est = normalise_estimator(estimator)
    slack_est = normalise_estimator(slack.estimator)
    slack_ne = slack.params.get("n_effective")
    if n_effective is not None and slack_ne is not None and abs(float(n_effective) - float(slack_ne)) > 1e-6:
        raise ValueError(
            "n_effective = %.6g in the report and %.6g in the slack." % (n_effective, slack_ne))
    bound_status = "available"
    if est != slack_est:
        bound_status = (
            "unavailable: the levels come from a %s and the slack prices a %s. The two differ by "
            "sqrt(n / n_eff); declare n_eff with effective_n() and re-price, or change the metric "
            "to the unweighted mean of per-record losses." % (est, slack_est))
        print("WARNING (estimator): " + bound_status)

    published = bound_status == "available"
    # n_eff is only inherited from a slack that prices the estimator the levels came from.
    n_eff = (float(n_effective) if n_effective is not None
             else (float(slack_ne) if published and slack_ne is not None else None))
    # The cumulative audit-derived transcript bits is a count, never absent: the declared budget
    # where one was priced, and zero for a panel that released nothing into training.
    if slack.transcript_bits is not None:
        bits = float(slack.transcript_bits)
    elif slack.provenance is not None:
        bits = float(slack.provenance.released_bits)
    else:
        bits = 0.0

    return {
        "unit": UNIT,
        "terminal_level": round(fn.terminal_level, 6),
        "signed_gain": round(fn.signed_gain, 6),
        "lower_bound_on_improvement": (round(fn.lower_bound, 6) if published else None),
        "discrimination_ratio": (round(fn.discrimination_ratio, 4)
                                 if published and math.isfinite(fn.discrimination_ratio) else None),
        "bound_status": bound_status,
        "target_version": target_version,
        "reference_version": reference_version,
        "masking_policy_version": masking_policy_version,
        "audit_unit": audit_unit,
        "audit_unit_kind": unit_kind,
        "n_independent_units": n_units,
        "estimator": est,
        "n_effective": n_eff,
        "n_effective_required_for_positive_bound": (
            None if published or slack.route != "c" else
            n_effective_for_positive_bound(gain=fn.signed_gain, R=float(slack.params["R"]),
                                           delta=float(slack.params["delta"]),
                                           N=float(slack.params.get("N", 1.0)))),
        "slack": round(fn.slack, 6),
        "slack_route": slack.route,
        "slack_route_label": slack.route_label,
        "slack_prices_estimator": slack_est,
        "slack_parameters": {k: (_finite_or_none(v) if isinstance(v, float) else v)
                             for k, v in slack.params.items()},
        "transcript_bits": bits,
        "commitment_record": dict(commitment_record),
        "levels": {
            "L_reference": round(fn.level_ref, 6),
            "L_initial": round(fn.level_initial, 6),
            "L_terminal": round(fn.level_terminal, 6),
        },
        "evidence_digest": digest,
        "notes": list(component_notes or []),
    }


_report_demo = audit_cp_report(
    _ref, _l0, _lT, _slack,
    target_version="toy-target/v1",
    reference_version="uniform-over-10/v1",
    masking_policy_version="full-transcript/v1",
    audit_unit="one synthetic audit example; the loss is the mean log loss over that example",
    n_independent_units=_n,
    commitment_record={
        "state_commitment": "sha256:0000000000000000000000000000000000000000000000000000000000000000",
        "committed_at": "1970-01-01T00:00:00Z",
        "challenge_drawn_after_commitment": True,
    },
    R=4.605170186,
)
print(json.dumps(_report_demo, indent=2)[:900] + "\n  ...")

print()
print("-- a report whose commitment is a hash of its own evidence:")
try:
    audit_cp_report(
        _ref, _l0, _lT, _slack,
        target_version="toy-target/v1", reference_version="uniform-over-10/v1",
        masking_policy_version="full-transcript/v1",
        audit_unit="one synthetic audit example", n_independent_units=_n, R=4.605170186,
        commitment_record={"state_commitment": evidence_digest(_ref, _l0, _lT),
                           "committed_at": "after scoring",
                           "challenge_drawn_after_commitment": True},
    )
except ValueError as exc:
    print("refused:", str(exc).split(". ")[0] + ".")

print()
print("-- a report published in tokens:")
try:
    audit_cp_report(
        _ref, _l0, _lT, _slack,
        target_version="toy-target/v1", reference_version="uniform-over-10/v1",
        masking_policy_version="full-transcript/v1",
        audit_unit="token", n_independent_units=_n, R=4.605170186,
        commitment_record={"state_commitment": "sha256:00", "committed_at": "t",
                           "challenge_drawn_after_commitment": True},
    )
except DependentUnitError as exc:
    print("refused:", str(exc).split(". ")[0] + ".")

## 7. Resolution before the run

The route (c) slack is fixed by the panel size, the loss range, the confidence level and the
declared candidate count, and by nothing about the model or the run. It is therefore available
before training begins, and a panel can be sized to the improvement it has to resolve.

`resolution_before_run(n, R, delta, N, unit_kind)` returns the slack a panel of that size will buy.
`panel_size_for_resolution()` inverts it. Both are sizing arithmetic and neither returns a `Slack`:
a number produced without a provenance cannot become the slack of a report.

In [ ]:
def resolution_before_run(*, n: int, R: float, delta: float, N: float,
                          unit_kind: str = "audit_unit",
                          estimator: str = UNWEIGHTED_MEAN,
                          n_effective: Optional[float] = None,
                          quiet: bool = False) -> Dict[str, float]:
    """Slack available from a panel of n units, before any training. Route (c) arithmetic.

    Returns a plain dict, not a Slack: sizing arithmetic carries no provenance, so its number
    cannot be handed to five_numbers() as though a panel had been declared.
    """
    Delta = panel_radius(n=n, R=R, delta=delta, N=N, unit_kind=unit_kind, estimator=estimator,
                         n_effective=n_effective, where="a resolution_before_run radius",
                         quiet=quiet)
    return {"n": int(n), "R": float(R), "delta": float(delta), "N": float(N),
            "unit_kind": normalise_unit_kind(unit_kind), "radius": Delta,
            "slack_if_sealed": 2.0 * Delta, "slack": 2.0 * Delta}


def panel_size_for_resolution(*, gamma_target: float, R: float, delta: float, N: float) -> int:
    """Smallest n whose route (c) slack is at most gamma_target."""
    if gamma_target <= 0:
        raise ValueError("gamma_target must be positive")
    Delta = gamma_target / 2.0
    n = R * R * math.log(2.0 * N / delta) / (2.0 * Delta * Delta)
    return int(math.ceil(n))


R_ARC, DELTA_ARC = 4.605170186, 0.05

print("Sizing a panel at R = %.6f (probability floor 0.01), delta = %.2f, N = 1." % (R_ARC, DELTA_ARC))
print()
print("  smallest improvement to resolve      required n")
for g in (0.50, 0.25, 0.154891, 0.10, 0.05, 0.02):
    need = panel_size_for_resolution(gamma_target=g, R=R_ARC, delta=DELTA_ARC, N=1)
    print("  %-36s %10d" % ("%.6f nats" % g, need))

print()
print("  panel size        radius Delta        slack Gamma")
for n in (200, 1024, 4096, 12288, 65536, 262144):
    r = resolution_before_run(n=n, R=R_ARC, delta=DELTA_ARC, N=1, quiet=True)
    print("  %-16d %14.6f %18.6f" % (n, r["radius"], r["slack_if_sealed"]))

print()
print("  Declared candidate set N at n = 12288:")
for label, N in (("1", 1.0), ("10^6", 1e6), ("10^9", 1e9), ("2^1000", 2.0 ** 1000)):
    r = resolution_before_run(n=12288, R=R_ARC, delta=DELTA_ARC, N=N, quiet=True)
    print("  N = %-10s radius %.6f   slack %.6f" % (label, r["radius"], r["slack_if_sealed"]))

_need = panel_size_for_resolution(gamma_target=0.10, R=R_ARC, delta=DELTA_ARC, N=1)
_res12288 = resolution_before_run(n=12288, R=R_ARC, delta=DELTA_ARC, N=1,
                                  quiet=True)["slack_if_sealed"]
print()
print("Worked sizing example. A benchmark that has to resolve 0.10 nats at R = %.6f, delta = %.2f"
      % (R_ARC, DELTA_ARC))
print("and N = 1 needs n = %d independent audit units. The 12,288-unit ARC-TGI panel resolves" % _need)
print("%.6f nats, so it separates a 0.74-nat gain and would not separate a 0.10-nat one." % _res12288)

print()
print("  the same 12,288-record panel, were its level a weighted mean (Section 3):")
for _ne in (12288, 8400, 6843, 1365, 289):
    _r = resolution_before_run(n=12288, R=R_ARC, delta=DELTA_ARC, N=1, quiet=True,
                               estimator=WEIGHTED_MEAN, n_effective=float(_ne))
    print("  n_eff = %-8d radius %.6f   slack %.6f" % (_ne, _r["radius"], _r["slack_if_sealed"]))

## 8. Reconciliation: increments against endpoints

Audit compression progress is a potential difference, `r_t = B_Q(p_t) - B_Q(p_{t-1})`, so the
increments telescope and a reported total is checked against the two endpoint levels alone. That is
what lets a third party check a total without seeing the trajectory: verification does not depend on
the number of steps, and the intermediate states never have to be disclosed.


In [ ]:
def reconcile_increments(increments, *, level_initial: float, level_terminal: float,
                         tol: float = 1e-9) -> Dict[str, Any]:
    """Check that per-step increments sum to the endpoint difference B_Q(p_T) - B_Q(p_0)."""
    r = np.asarray(increments, dtype=np.float64)
    if r.ndim != 1:
        raise ValueError("increments must be a 1-D vector of per-step values")
    if not np.all(np.isfinite(r)):
        raise ValueError("increments contain non-finite entries")
    total = float(r.sum())
    endpoint = float(level_terminal) - float(level_initial)
    residual = total - endpoint
    ok = abs(residual) <= tol * max(1.0, abs(endpoint))
    out = {"steps": int(r.size), "sum_of_increments": total, "endpoint_difference": endpoint,
           "residual": residual, "reconciles": bool(ok),
           "negative_increments": int(np.sum(r < 0.0))}
    status = "reconciles" if ok else "DOES NOT RECONCILE"
    print("%s: %d steps, sum %+.9f against endpoint difference %+.9f, residual %+.2e"
          % (status, out["steps"], total, endpoint, residual))
    return out


_rng2 = np.random.default_rng(7)
_G_true = 0.73658
_steps = _rng2.normal(0.0, 0.05, 400)
_steps += (_G_true - _steps.sum()) / _steps.size
_L_ref, _L0v, _LTv = math.log(11.0), 2.48413, 1.74755
_B0, _BT = _L_ref - _L0v, _L_ref - _LTv

print("A 400-step trajectory, %d of whose increments are negative:" % int(np.sum(_steps < 0)))
_rec = reconcile_increments(_steps, level_initial=_B0, level_terminal=_BT)
print()
print("Same ledger with one increment overstated by 0.05 nats:")
_tampered = _steps.copy()
_tampered[123] += 0.05
_bad = reconcile_increments(_tampered, level_initial=_B0, level_terminal=_BT)
print()
print("A third party checks the total against the two endpoint levels alone, so a trajectory can be")
print("verified without being disclosed and the check does not grow with its length.")
assert _rec["reconciles"] and not _bad["reconciles"]

## 9. Worked example: the ARC-TGI run from the paper

The paper's worked example is the ARC-TGI curriculum run, scored on the sealed panel that training
never touched. Its declared parameters are

```
n = 12288 audit records     R = 4.605170186 = -ln 0.01     delta = 0.05     N = 1     route (c)
estimator = content-cell-weighted ratio, priced at n_eff = 6521.8
```

and the published values are `Delta = 0.077445`, `Gamma = 0.154891`, mean `L_0 = 2.4841`, mean
`L_T = 1.7476`, mean `G = 0.7366`, lower bound `0.5817`, discrimination ratio `4.76`, with per-seed
lower bounds from `0.4844` to `0.6561`, positive in 20 of 20 seeds.

The per-seed endpoint levels and the per-seed cell counts come from the record shipped with this
repository, `experiments/results_data/c1_panelB_initial_loss.json`, regenerated by
`experiments/c1_panelB_initial_loss.py`. For each seed it holds the initial and terminal audit level
of panel B, the sealed 12,288-record panel scored once after the final training step and disjoint
from the probe panel that the per-step scheduler reward read, together with `sum_cells`,
`sum_cells_sq` and `n_effective` for that seed's panel.

**What the estimator is and what prices it.** `audit_panel_loss` in `experiments/cp_models.py`
returns

```
floored_ce = (total floored CE over every content cell of the panel) / (total content cells)
```

which is `sum_i w_i m_i` with `w_i = c_i / sum_j c_j` for the content-cell count `c_i` of record
`i`, and ARC-TGI grids run from 1x1 to 30x30 inside the padded canvas, so the `c_i` vary. Hoeffding
for a weighted mean of bounded terms has radius `R * sqrt(sum_i w_i^2 * ln(2N/delta) / 2)`, which is
the unweighted radius with `n_eff = (sum_i c_i)^2 / sum_i c_i^2` in place of `n`. Section 9.1 puts
the unweighted radius beside it and shows why that radius is the wrong one for this estimator.

**What this section reproduces.** The five numbers depend on the loss vectors only through their
means, so vectors carrying the shipped levels reproduce the published arithmetic exactly; the
per-unit dispersion constructed below enters nothing. The terminal level `B` also needs a benchmark
reference, which the published values do not fix. We declare the uniform quote over the 11 ARC-TGI
colour classes, `L_Q(p_ref) = ln 11 = 2.397895` nats, and report `B` against it.


In [ ]:
VALIDATION_TARGET = {
    "n": 12288, "R": 4.605170186, "delta": 0.05, "N": 1, "route": "c",
    "estimator": WEIGHTED_MEAN, "n_effective": 6521.8,
    "Delta": 0.077445, "Gamma": 0.154891,
    "mean_L0": 2.4841, "mean_LT": 1.7476, "mean_G": 0.7366,
    "lower_bound": 0.5817, "discrimination_ratio": 4.76,
    "per_seed_lower_bound_min": 0.4844, "per_seed_lower_bound_max": 0.6561,
    "seeds_positive": 20, "seeds": 20,
}


def matches_published(have: float, want: float) -> bool:
    """Compare at the precision the published value is quoted to."""
    text = repr(want)
    ndp = len(text.split(".")[1]) if "." in text else 0
    return round(float(have), ndp) == want


def _find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "experiments").is_dir() and (p / "README.md").is_file():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd().resolve())

SEED_FILE_DIRS = [
    REPO_ROOT / "experiments" / "results_data",
    REPO_ROOT / "experiments" / "data",
    REPO_ROOT / "experiments",
    REPO_ROOT / "data",
    REPO_ROOT / "results",
    REPO_ROOT / "notebooks" / "data",
]
_L0_ALIASES = ("l0", "l_0", "loss_0", "loss0", "initial_loss", "l_initial", "mean_l0", "l0_mean",
               "audit_loss_initial", "floored_ce_initial", "ce_initial", "ce_0", "initial")
_LT_ALIASES = ("lt", "l_t", "loss_t", "losst", "terminal_loss", "l_terminal", "mean_lt", "lt_mean",
               "audit_loss_terminal", "floored_ce_terminal", "ce_terminal", "ce_t", "terminal",
               "final_audit_loss")
_SEED_ALIASES = ("seed", "seed_id", "run", "run_id", "replicate")


def _pick(keys, aliases):
    lowered = {str(k).strip().lower(): k for k in keys}
    for a in aliases:
        if a in lowered:
            return lowered[a]
    return None


def _rows_to_arrays(rows):
    """Turn a list of per-seed dicts into (seeds, L0, LT), dropping rows not marked ok."""
    rows = [r for r in rows if isinstance(r, dict)]
    rows = [r for r in rows if str(r.get("status", "ok")).lower() == "ok"]
    if len(rows) < 2:
        return None
    k0 = _pick(rows[0].keys(), _L0_ALIASES)
    kT = _pick(rows[0].keys(), _LT_ALIASES)
    if k0 is None or kT is None:
        return None
    ks = _pick(rows[0].keys(), _SEED_ALIASES)
    try:
        seeds = [int(float(r[ks])) for r in rows] if ks else list(range(len(rows)))
        L0 = [float(r[k0]) for r in rows]
        LT = [float(r[kT]) for r in rows]
    except (KeyError, TypeError, ValueError):
        return None
    order = np.argsort(np.array(seeds))
    return (np.array(seeds, dtype=int)[order],
            np.array(L0, dtype=np.float64)[order],
            np.array(LT, dtype=np.float64)[order])


def _row_blocks(obj):
    """Yield the plausible per-seed row lists inside a parsed JSON document."""
    if isinstance(obj, list):
        yield obj
        return
    if not isinstance(obj, dict):
        return
    for key in ("per_seed", "seeds", "rows", "results", "per_seed_endpoints", "data"):
        block = obj.get(key)
        if isinstance(block, list):
            yield block
        elif isinstance(block, dict):
            yield [dict(r, seed=r.get("seed", k))
                   for k, r in block.items() if isinstance(r, dict)]


def _load_candidate(path: Path):
    try:
        if path.suffix.lower() == ".csv":
            import csv
            with path.open(newline="") as fh:
                return _rows_to_arrays(list(csv.DictReader(fh)))
        if path.suffix.lower() == ".json":
            obj = json.loads(path.read_text())
            for block in _row_blocks(obj):
                got = _rows_to_arrays(block)
                if got is not None:
                    return got
            if isinstance(obj, dict):
                k0, kT = _pick(obj.keys(), _L0_ALIASES), _pick(obj.keys(), _LT_ALIASES)
                if k0 and kT and isinstance(obj[k0], list):
                    ks = _pick(obj.keys(), _SEED_ALIASES)
                    seeds = (np.array([int(s) for s in obj[ks]], dtype=int) if ks
                             else np.arange(len(obj[k0])))
                    return (seeds, np.asarray(obj[k0], dtype=np.float64),
                            np.asarray(obj[kT], dtype=np.float64))
    except Exception as exc:
        print("  skipped %s (%s)" % (path.name, exc))
    return None


def find_seed_endpoint_file():
    """Find the shipped per-seed endpoint record: per-seed initial and terminal audit levels."""
    for d in SEED_FILE_DIRS:
        if not d.is_dir():
            continue
        for path in sorted(d.glob("*")):
            if path.suffix.lower() not in (".csv", ".json"):
                continue
            got = _load_candidate(path)
            if got is not None and got[1].size >= 2:
                meta = {}
                if path.suffix.lower() == ".json":
                    obj = json.loads(path.read_text())
                    if isinstance(obj, dict):
                        meta = obj
                return got + (str(path.relative_to(REPO_ROOT)), meta)
    return None


def load_arc_tgi_endpoints():
    """Return (seeds, L0, LT, source, metadata) from the shipped per-seed endpoint record."""
    print("repository root:", REPO_ROOT)
    found = find_seed_endpoint_file()
    if found is None:
        raise FileNotFoundError(
            "no per-seed endpoint record found under:\n    "
            + "\n    ".join(str(d) for d in SEED_FILE_DIRS)
            + "\nThe ARC-TGI record ships as experiments/results_data/c1_panelB_initial_loss.json"
              " and is regenerated by experiments/c1_panelB_initial_loss.py."
        )
    seeds, L0, LT, src, meta = found
    print("per-seed endpoints: %s (%d seeds)" % (src, seeds.size))
    for key in ("measurement", "generated_by"):
        if key in meta:
            print("  %-13s %s" % (key + ":", meta[key]))
    panel = meta.get("panel", {}) if isinstance(meta, dict) else {}
    if panel:
        print("  panel:        %s (sealed=%s, scored_during_training=%s)"
              % (panel.get("name"), panel.get("sealed"), panel.get("scored_during_training")))
    loss = meta.get("loss", {}) if isinstance(meta, dict) else {}
    if loss:
        print("  loss:         %s" % loss.get("name"))
        print("  range R:      %.9f" % float(loss.get("range_R", float("nan"))))
    return seeds, L0, LT, src, meta


ARC_SEEDS, ARC_L0, ARC_LT, ARC_SOURCE, ARC_META = load_arc_tgi_endpoints()
print()
print("seeds: %d   mean L_0 = %.6f   mean L_T = %.6f"
      % (ARC_SEEDS.size, ARC_L0.mean(), ARC_LT.mean()))

In [ ]:
ARC_N = VALIDATION_TARGET["n"]
ARC_R = VALIDATION_TARGET["R"]
ARC_DELTA = VALIDATION_TARGET["delta"]
ARC_NCAND = VALIDATION_TARGET["N"]
ARC_L_REF = math.log(11.0)          # uniform quote over the 11 ARC-TGI colour classes


def losses_with_mean(target_mean: float, n: int, R: float, spread: float, rng) -> np.ndarray:
    """A per-unit loss vector of length n whose mean is exactly target_mean and whose entries lie
    in [0, R]. Only the mean enters the five numbers; the dispersion is illustrative."""
    g = rng.standard_normal(n)
    g = (g - g.mean()) / g.std()
    v = target_mean + spread * g
    if v.min() < 0.0 or v.max() > R:
        raise ValueError("spread too large for the declared loss range")
    return v


arc_panel = declare_panel(
    "ARC-TGI sealed audit panel",
    reward=False,
    early_stopping=False,
    checkpoint_selection=False,
    hyperparameter_selection=False,
    data_selection=False,
    note=("held out for the whole run; the per-step scheduler reward came from a separate "
          "512-example probe, priced under route (d) in Section 10"),
)
print(arc_panel.describe())
# Cross-check the declaration against the shipped record. The declaration above is the user's, not
# the file's: reading a provenance out of a data file would be the bypass this guardrail exists to
# prevent. What the record can do is contradict the declaration.
_rec_panel = ARC_META.get("panel", {}) if isinstance(ARC_META, dict) else {}
if _rec_panel:
    if _rec_panel.get("sealed") is not True or _rec_panel.get("scored_during_training") is not False:
        raise PanelSteeredError(
            "the shipped record contradicts the declaration above: panel %r has sealed=%r and "
            "scored_during_training=%r, so route (c) does not apply."
            % (_rec_panel.get("name"), _rec_panel.get("sealed"),
               _rec_panel.get("scored_during_training")))
    print("record cross-check: panel %r sealed, not scored during training"
          % _rec_panel.get("name"))

_rec_units = set()
for _blk in _row_blocks(ARC_META):
    for _r in _blk:
        if isinstance(_r, dict) and "n_panel" in _r:
            _rec_units.add(int(_r["n_panel"]))
if _rec_units:
    assert _rec_units == {ARC_N}, \
        "record panel sizes %r disagree with the declared n = %d" % (sorted(_rec_units), ARC_N)
    print("record cross-check: every seed scored n = %d independent audit records" % ARC_N)

_rec_R = float(ARC_META.get("loss", {}).get("range_R", ARC_R)) if isinstance(ARC_META, dict) else ARC_R
assert abs(_rec_R - ARC_R) < 1e-6, \
    "record loss range %.9f disagrees with the declared R = %.9f" % (_rec_R, ARC_R)
print("record cross-check: loss range R = %.9f" % _rec_R)

# What estimator produced the levels, and the sample size that prices it. Both come from the
# record: the estimator is a content-cell-weighted ratio, and n_eff = (sum c_i)^2 / sum c_i^2 is
# stored per seed from the regenerated panel, with its mean over seeds under loss.n_effective.
ARC_ESTIMATOR = normalise_estimator(ARC_META["loss"]["estimator"])
assert ARC_ESTIMATOR == VALIDATION_TARGET["estimator"], ARC_ESTIMATOR
_per_seed_rows = ARC_META.get("per_seed", {})
ARC_NEFF_BY_SEED = {}
for _s in ARC_SEEDS:
    _r = _per_seed_rows.get(str(int(_s)), {})
    if _r.get("n_effective") is None:
        raise WeightedEstimatorError(
            "seed %d carries no n_effective in %s; the archived level is a weighted mean and "
            "cannot be priced without it. Re-run experiments/c1_panelB_initial_loss.py --force."
            % (int(_s), ARC_SOURCE))
    ARC_NEFF_BY_SEED[int(_s)] = float(_r["n_effective"])
ARC_N_EFFECTIVE = float(ARC_META["loss"]["n_effective"])          # mean over seeds: the headline
assert abs(ARC_N_EFFECTIVE - float(np.mean(list(ARC_NEFF_BY_SEED.values())))) < 1e-6
print("record cross-check: estimator = %s, n_effective = %.1f (per seed %.1f to %.1f)"
      % (ARC_ESTIMATOR, ARC_N_EFFECTIVE, min(ARC_NEFF_BY_SEED.values()),
         max(ARC_NEFF_BY_SEED.values())))
print()
check_independent_units(ARC_N, "audit_unit", n_records=ARC_N)
print()

# The published slack: route (c) for the weighted estimator, priced at the mean n_eff.
arc_slack = compute_slack("c", provenance=arc_panel, n=ARC_N, R=ARC_R, delta=ARC_DELTA,
                          N=ARC_NCAND, unit_kind="audit_unit", estimator=ARC_ESTIMATOR,
                          n_effective=ARC_N_EFFECTIVE)
arc_Delta = arc_slack.params["Delta"]
print("route (c) at n = %d records, n_eff = %.1f, R = %.9f, delta = %.2f, N = %d"
      % (ARC_N, ARC_N_EFFECTIVE, ARC_R, ARC_DELTA, ARC_NCAND))
print("  radius Delta = %.6f nats" % arc_Delta)
print("  slack  Gamma = %.6f nats" % arc_slack.gamma)
print("  prices       = %s at n_eff" % arc_slack.estimator)

# Per seed: each panel is its own draw and is priced at its own n_eff.
_rng_arc = np.random.default_rng(20260101)
loss_ref_arc = np.full(ARC_N, ARC_L_REF)
per_seed = []
for s, l0, lt in zip(ARC_SEEDS, ARC_L0, ARC_LT):
    v0 = losses_with_mean(float(l0), ARC_N, ARC_R, 0.30, _rng_arc)
    vT = losses_with_mean(float(lt), ARC_N, ARC_R, 0.30, _rng_arc)
    _sl = compute_slack("c", provenance=arc_panel, n=ARC_N, R=ARC_R, delta=ARC_DELTA,
                        N=ARC_NCAND, unit_kind="audit_unit", estimator=ARC_ESTIMATOR,
                        n_effective=ARC_NEFF_BY_SEED[int(s)], quiet=True)
    per_seed.append((int(s), ARC_NEFF_BY_SEED[int(s)], _sl.gamma,
                     five_numbers(loss_ref_arc, v0, vT, _sl, R=ARC_R)))

print()
print("  seed     n_eff     Gamma       L_0       L_T         G     G - Gamma    G / Gamma")
for s, ne, gam, fn in per_seed:
    print("  %4d  %8.1f  %8.6f  %8.4f  %8.4f  %8.4f    %8.4f     %8.2f"
          % (s, ne, gam, fn.level_initial, fn.level_terminal, fn.signed_gain, fn.lower_bound,
             fn.discrimination_ratio))

per_seed_lb = np.array([fn.lower_bound for _, _, _, fn in per_seed])
print()
print("  per-seed lower bound: min %.4f, max %.4f, positive in %d of %d seeds"
      % (per_seed_lb.min(), per_seed_lb.max(), int((per_seed_lb > 0).sum()), per_seed_lb.size))


In [ ]:
# Pooled report: the panel scored at the mean initial and mean terminal level over the 20 seeds,
# priced at the mean n_eff.
loss_0_arc = losses_with_mean(float(ARC_L0.mean()), ARC_N, ARC_R, 0.30, _rng_arc)
loss_T_arc = losses_with_mean(float(ARC_LT.mean()), ARC_N, ARC_R, 0.30, _rng_arc)
arc_five = five_numbers(loss_ref_arc, loss_0_arc, loss_T_arc, arc_slack, R=ARC_R)

print("ARC-TGI, mean over %d seeds, scored on the sealed panel" % ARC_SEEDS.size)
print("  L_Q(p_ref) = %.6f   L_Q(p_0) = %.6f   L_Q(p_T) = %.6f"
      % (arc_five.level_ref, arc_five.level_initial, arc_five.level_terminal))
print()
print(arc_five.summary())

# ------------------------------------------- assertions, at the precision each value is quoted to
V = VALIDATION_TARGET
_checks = [
    ("n_eff", ARC_N_EFFECTIVE, V["n_effective"]),
    ("Delta", arc_Delta, V["Delta"]),
    ("Gamma", arc_slack.gamma, V["Gamma"]),
    ("mean L_0", arc_five.level_initial, V["mean_L0"]),
    ("mean L_T", arc_five.level_terminal, V["mean_LT"]),
    ("mean G", arc_five.signed_gain, V["mean_G"]),
    ("lower bound", arc_five.lower_bound, V["lower_bound"]),
    ("ratio", arc_five.discrimination_ratio, V["discrimination_ratio"]),
    ("per-seed lb min", per_seed_lb.min(), V["per_seed_lower_bound_min"]),
    ("per-seed lb max", per_seed_lb.max(), V["per_seed_lower_bound_max"]),
]
for _name, _have, _want in _checks:
    assert matches_published(_have, _want), "%s: have %.9f, published %r" % (_name, _have, _want)
assert ARC_SEEDS.size == V["seeds"], "seed count: %d" % ARC_SEEDS.size
assert int((per_seed_lb > 0).sum()) == V["seeds_positive"]

# The reported total is checked against the two endpoint levels alone.
_arc_rec = reconcile_increments(
    np.full(5000, arc_five.signed_gain / 5000.0),
    level_initial=arc_five.level_ref - arc_five.level_initial,
    level_terminal=arc_five.level_ref - arc_five.level_terminal,
)
assert _arc_rec["reconciles"]

print()
print("VALIDATION PASSED against the published ARC-TGI values.")
for _name, _have, _want in _checks:
    print("  %-16s %-14.6f (published %r)" % (_name, _have, _want))
print("  %-16s %d/%d" % ("positive seeds", int((per_seed_lb > 0).sum()), V["seeds"]))
print("  data source      %s" % ARC_SOURCE)
print()
print("Those are the published numbers and the arithmetic that produced them: the weighted panel")
print("level priced at n_eff = %.1f of its %d records. Section 9.1 puts the unweighted radius at n"
      % (ARC_N_EFFECTIVE, ARC_N))
print("beside it and shows why it does not apply.")


### 9.1 The unweighted radius beside the one that applies

The archived metric is the content-cell-weighted ratio, `sum_i w_i m_i` with `w_i = c_i / sum_j c_j`.
Conditional on the counts it is a weighted mean of 12,288 independent terms in `[0, R]`, and
Hoeffding gives it radius `R * sqrt(sum_i w_i^2 * ln(2N/delta) / 2)`: the unweighted radius at `n`
inflated by `sqrt(n / n_eff)` with `n_eff = 1 / sum_i w_i^2`. Cauchy-Schwarz puts `n_eff <= n`, with
equality only when every record carries the same number of cells, so the correction has a fixed
sign. On this panel `n_eff` is 6,521.8 of 12,288 records and the inflation is 1.373.

The cell counts are recoverable exactly because panel B is a deterministic function of the seed,
and the regenerated panel reproduces the archived initial level to the last bit, which is what
licenses attaching the regenerated `n_eff` to the archived terminal level. The record ships
`sum_cells`, `sum_cells_sq` and `n_effective` for every seed.

The guardrails refuse the mismatched pairing rather than warning about it: `slack_fixed_panel`
raises when a weighted estimator is declared without `n_eff`, and `audit_cp_report` publishes no
bound when the slack prices a different estimator from the one the levels came from. The cell below
shows both refusals, then the threshold: the bound stays positive for every `n_eff` above
`2 R^2 ln(2N/delta) / G^2`, which is 289 here, and the measured value is more than twenty times that.


In [ ]:
# (i) The weighted estimator declared without n_eff is refused, not warned about.
print("-- the archived estimator declared without n_eff:")
try:
    compute_slack("c", provenance=arc_panel, n=ARC_N, R=ARC_R, delta=ARC_DELTA, N=ARC_NCAND,
                  unit_kind="audit_unit", estimator=ARC_ESTIMATOR)
    raise AssertionError("the weighted declaration should have been refused")
except WeightedEstimatorError as exc:
    print("  refused:", str(exc).split(". ")[0] + ".")

# (ii) The unweighted radius at n, for reference. It prices an unweighted mean of n records, which
# the archived level is not. Paired with these levels, the report publishes no bound.
_arc_slack_unweighted = compute_slack("c", provenance=arc_panel, n=ARC_N, R=ARC_R, delta=ARC_DELTA,
                                      N=ARC_NCAND, unit_kind="audit_unit",
                                      estimator=UNWEIGHTED_MEAN, quiet=True)
_infl = math.sqrt(ARC_N / ARC_N_EFFECTIVE)
print()
print("-- the unweighted radius at n = %d, for reference only:" % ARC_N)
print("  Delta = %.6f   Gamma = %.6f   inflation to the weighted radius sqrt(n/n_eff) = %.3f"
      % (_arc_slack_unweighted.params["Delta"], _arc_slack_unweighted.gamma, _infl))
assert abs(_arc_slack_unweighted.gamma * _infl - arc_slack.gamma) < 1e-9
_mismatched = audit_cp_report(
    loss_ref_arc, loss_0_arc, loss_T_arc, _arc_slack_unweighted,
    target_version="arc-tgi/sealed-audit-panel/v1",
    reference_version="uniform-over-11-colour-classes/v1",
    masking_policy_version="arc-tgi/content-cells-pad-excluded/v1",
    audit_unit=("one ARC-TGI panel item, a padded 30x30 grid; the archived panel level is the "
                "total probability-floored cross-entropy at floor 0.01 over every content cell of "
                "the panel, divided by the total content-cell count, so it weights each item by "
                "its content-cell count rather than averaging per-item values"),
    estimator=ARC_ESTIMATOR,
    n_effective=None,
    n_independent_units=ARC_N,
    R=ARC_R,
    commitment_record={
        "state_commitment": None,
        "committed_at": None,
        "challenge_drawn_after_commitment": False,
        "what_the_archive_holds": (
            "endpoint audit levels for 20 seeds; no checkpoint hash was recorded, so nothing here "
            "fixes theta_T before panel B was scored"),
        "panel_drawn_from": (
            "panel(seed + 2000, 512, 30, LEARNABLE_TASKS), a deterministic function of the run "
            "seed fixed in the configuration before training"),
        "note": "reported as absent rather than substituted",
    },
)
print("  paired with the weighted levels the report says: bound_status = %s"
      % _mismatched["bound_status"].split(".")[0])
assert _mismatched["bound_status"].startswith("unavailable")
assert _mismatched["lower_bound_on_improvement"] is None

# (iii) The threshold, and how far above it the measured n_eff sits.
ARC_G = float(arc_five.signed_gain)
ARC_NEFF_MIN = n_effective_for_positive_bound(gain=ARC_G, R=ARC_R, delta=ARC_DELTA, N=ARC_NCAND)
print()
print("Threshold at G = %.6f nats: the bound is positive for every n_eff > %d of n = %d records."
      % (ARC_G, ARC_NEFF_MIN, ARC_N))
print("  The measured n_eff = %.1f is %.1f times that." % (ARC_N_EFFECTIVE, ARC_N_EFFECTIVE / ARC_NEFF_MIN))
assert ARC_N_EFFECTIVE > ARC_NEFF_MIN
# the mismatched report above withheld its bound, so it carries this threshold
assert _mismatched["n_effective_required_for_positive_bound"] == ARC_NEFF_MIN


### 9.2 The public report object for this run

The report declares the estimator that produced the levels, the weighted one, and the slack prices
that estimator at the record's `n_eff`, so the object publishes the levels, the gain, the slack, the
lower bound and the ratio, with `bound_status` available and `n_effective` carried explicitly.

Its commitment record is the one thing this artifact cannot supply. The archived run stored
endpoint audit levels, not checkpoint hashes, so nothing in it fixes `theta_T` before panel B was
drawn. The record says so and takes the warning. A digest of the loss vectors would be refused if it
were offered in that field: it is computable only after scoring, so it commits to nothing.


In [ ]:
arc_report = audit_cp_report(
    loss_ref_arc, loss_0_arc, loss_T_arc, arc_slack,
    target_version="arc-tgi/sealed-audit-panel/v1",
    reference_version="uniform-over-11-colour-classes/v1",
    masking_policy_version="arc-tgi/content-cells-pad-excluded/v1",
    audit_unit=("one ARC-TGI panel item, a padded 30x30 grid; the archived panel level is the "
                "total probability-floored cross-entropy at floor 0.01 over every content cell of "
                "the panel, divided by the total content-cell count, so it weights each item by "
                "its content-cell count rather than averaging per-item values"),
    estimator=ARC_ESTIMATOR,
    n_effective=ARC_N_EFFECTIVE,
    n_independent_units=ARC_N,
    R=ARC_R,
    commitment_record={
        "state_commitment": None,
        "committed_at": None,
        "challenge_drawn_after_commitment": False,
        "what_the_archive_holds": (
            "endpoint audit levels for 20 seeds; no checkpoint hash was recorded, so nothing here "
            "fixes theta_T before panel B was scored"),
        "panel_drawn_from": (
            "panel(seed + 2000, 512, 30, LEARNABLE_TASKS), a deterministic function of the run "
            "seed fixed in the configuration before training"),
        "note": "reported as absent rather than substituted",
    },
    component_notes=[
        "scored on the sealed panel; the per-step scheduler reward used a separate 512-example probe",
        "the archived level is a content-cell-weighted ratio; its slack is priced at n_eff = %.1f "
        "of %d records, measured from the regenerated panel, which reproduces the archived initial "
        "level to the last bit" % (ARC_N_EFFECTIVE, ARC_N),
        "per-seed lower bounds %.4f to %.4f at each seed's own n_eff, positive in %d of %d seeds"
        % (per_seed_lb.min(), per_seed_lb.max(), int((per_seed_lb > 0).sum()), per_seed_lb.size),
        "data source: %s" % ARC_SOURCE,
    ],
)
print(json.dumps(arc_report, indent=2))

assert arc_report["bound_status"] == "available", arc_report["bound_status"]
assert abs(arc_report["lower_bound_on_improvement"] - arc_five.lower_bound) < 1e-6
assert abs(arc_report["discrimination_ratio"] - arc_five.discrimination_ratio) < 1e-4
assert abs(arc_report["n_effective"] - ARC_N_EFFECTIVE) < 1e-6
# the threshold is reported only when a bound is withheld; here the bound is published
assert arc_report["n_effective_required_for_positive_bound"] is None
assert arc_report["transcript_bits"] == 0.0
assert arc_report["commitment_record"]["state_commitment"] != arc_report["evidence_digest"]


## 10. The guardrail in action: the same run, the other panel

The same ARC-TGI run also has a second panel. The per-step scheduler reward was computed on a
separate 512-example probe, reused at every one of the 5,000 rounds, and its scores steered the
scheduler. A gain measured there is not a fixed-model measurement. Route (c) refuses it; route (d)
prices it against the bits released.


In [ ]:
steering_probe = declare_panel(
    "ARC-TGI 512-example scheduler probe",
    reward=True,                     # its scores were the per-step scheduler reward
    early_stopping=False,
    checkpoint_selection=False,
    hyperparameter_selection=False,
    data_selection=True,             # the scheduler chose which task family to train on next
    transcript_bits=5000 * 32.0,     # one float32 score released at each of 5000 rounds
    note="reused at every round; scores steered the training scheduler",
)
print(steering_probe.describe())
print()

try:
    compute_slack("c", provenance=steering_probe, n=512, R=ARC_R, delta=ARC_DELTA, N=1,
                  unit_kind="audit_unit")
    raise AssertionError("route (c) should have been refused")
except PanelSteeredError as exc:
    print(exc)

probe_slack = compute_slack("d", provenance=steering_probe, n=512, R=ARC_R, delta=ARC_DELTA,
                            unit_kind="audit_unit")
print()
print("route (d) at n = 512, B = %g bits:" % steering_probe.transcript_bits)
print("  " + str(probe_slack))


def transcript_budget_for_gain(*, gain: float, n: int, R: float, delta: float) -> float:
    """Largest transcript budget B, in bits, at which route (d) still leaves a positive bound."""
    lhs = n * (gain / R) ** 2 / 2.0 - math.log(1.0 / delta)
    return lhs / math.log(2.0)


print()
print("  transcript budget B (bits)      slack Gamma (nats)")
for B in (1, 4, 8, 64, 512, 4096, 160000):
    # A hypothetical probe that released exactly B bits, so the row is a declaration and not a
    # re-pricing of the archived probe below its own declared count.
    hypothetical = declare_panel(
        "ARC-TGI scheduler probe releasing %d bits" % B,
        reward=True, early_stopping=False, checkpoint_selection=False,
        hyperparameter_selection=False, data_selection=True, transcript_bits=float(B))
    s = compute_slack("d", provenance=hypothetical, n=512, R=ARC_R, delta=ARC_DELTA,
                      unit_kind="audit_unit", transcript_bits=float(B), quiet=True)
    verdict = "positive bound" if arc_five.signed_gain > s.gamma else "no positive bound"
    print("  %-30d %8.4f    %s" % (B, s.gamma, verdict))

B_max = transcript_budget_for_gain(gain=arc_five.signed_gain, n=512, R=ARC_R, delta=ARC_DELTA)
print()
print("At n = 512 and a gain of %.4f nats, route (d) leaves a positive lower bound only up to"
      % arc_five.signed_gain)
print("B = %.2f bits released across the whole run. The archived probe released about %g bits, so"
      % (B_max, steering_probe.transcript_bits))
print("it supports no lower bound. The same model and the same training run therefore support a")
print("lower bound on the sealed panel and none at all on the probe.")
assert probe_slack.gamma > arc_five.signed_gain

## 11. Optional: producing the loss vectors from a torch model

The core above is numpy-only and never imports torch. This section is a recipe for one common way of
producing the three vectors it wants. The function is defined and not called, and its import sits
inside the body, so this notebook runs with or without torch installed.

Two things to get right. Score the **same** panel in the **same** order under all three quotes, and
apply the probability floor before taking the logarithm so the losses respect the declared range `R`.
Reduce to one value per independent audit unit: if the model emits per-token or per-cell losses,
average them **within** each unit and return one number per unit. Do not return the flattened
per-token vector, and do not pool: dividing the panel's total loss by its total token count is the
weighted estimator of Section 3 and needs an `n_eff`. The recipe below deliberately does neither.


In [ ]:
def per_unit_losses_from_torch(model, batches, *, floor: float = 0.01, device: str = "cpu"):
    """OPTIONAL RECIPE, not part of the core and not called by this notebook.

    Returns a 1-D numpy vector of length n_units holding -log p(y|x) under `model`, one entry per
    independent audit unit, floored at `floor` so the loss respects R = -log(floor).

    `batches` yields (x, y, unit_id) where unit_id assigns every scored position to its audit unit
    (its example, document or episode). Positions sharing a unit_id are averaged together, which is
    what keeps n the count of independent units rather than of tokens or cells.
    """
    import torch                       # local import: the core never depends on torch
    import torch.nn.functional as F

    sums: Dict[int, float] = {}
    counts: Dict[int, int] = {}
    model.eval().to(device)
    with torch.no_grad():
        for x, y, unit_id in batches:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = logits.softmax(dim=-1)
            probs = (1.0 - floor) * probs + floor / probs.shape[-1]     # full-support quote
            p_y = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)
            nll = (-p_y.clamp_min(floor).log()).reshape(-1).cpu().numpy()
            uid = np.asarray(unit_id).reshape(-1)
            for u, v in zip(uid.tolist(), nll.tolist()):
                sums[u] = sums.get(u, 0.0) + v
                counts[u] = counts.get(u, 0) + 1

    units = sorted(sums)
    return np.array([sums[u] / counts[u] for u in units], dtype=np.float64)


print("per_unit_losses_from_torch is defined and not called.")
print("torch imported by this notebook:", "torch" in sys.modules)
print()
print("Usage sketch:")
print("  loss_ref = per_unit_losses_from_torch(reference_model, panel_batches())")
print("  loss_0   = per_unit_losses_from_torch(initial_model,   panel_batches())")
print("  loss_T   = per_unit_losses_from_torch(terminal_model,  panel_batches())")
print("  slack    = compute_slack('c', provenance=panel, n=len(loss_0), R=-math.log(0.01),")
print("                           delta=0.05, N=1, unit_kind='document')")
print("  report   = audit_cp_report(loss_ref, loss_0, loss_T, slack, ...)")

## 12. What the report does not say

The bound is restricted to the declared probabilistic target, a bounded or verifier-controlled loss,
authenticated causal order, and audit-derived channels that are actually included in the transcript.
Target misspecification, omitted side channels, alteration of the masking policy, correlated verifier
compromise, and objectives with no predictive encoding all sit outside it.

A gain measured under target version `v` is a statement about `Q_v`. When the target changes, close
the ledger and open the next one under the new version; a bridge state scored under both versions
separates the loss shift caused by the revision from progress under either target, in the same unit.

The route (c) and route (d) terms are conservative by construction. A negative or small result at a
given panel size is a statement about resolution, not about the system.
